# 1 · Conformer Ensemble Generator

**Chameleon Predictor reproducibility notebook — 1 of 3**  
*generate → calculate → plot*

**Paste any SMILES, run, and get 3D conformer ensembles in implicit water and chloroform (membrane mimic).** The CREST ensemble is reduced by RMSD to a diverse set of **unique conformers** (default 20 per solvent), so downstream analysis summarizes geometry without overweighting near-duplicate structures.

| | |
|---|---|
| **You provide** | a SMILES string (+ optional name, charge, threads, output dir) |
| **You get** | `water/ensemble.{sdf,json}` and `mem/ensemble.{sdf,json}` (unique-conformer set) |
| **Powered by** | `scripts/crest_conformers_standalone.generate_ensembles()` |

Energies and Boltzmann weights are still computed and stored, but as **metadata / QC** — geometry is the object of analysis in Notebooks 2–3, not thermodynamic weighting.

## Requirements — read first

The conformer search runs two **external programs** that are not Python packages:

- **`xtb`** and **`crest` 2.12** must be installed and on your `PATH`.
- Install: `conda install -c conda-forge xtb crest` (Linux / macOS / WSL — CREST has no native Windows build), or load your cluster's modules.
- Python packages needed: **`rdkit`**, `numpy`, `pandas` (all in the `base` env here).

**The intended workflow:** install the two binaries → restart the kernel → paste a SMILES → run. Because `xtb`/`crest` are external, the notebook can't be *fully* self-contained; if they are missing it will tell you exactly what to install rather than fail cryptically, and the final inspection cell falls back to a pre-computed example so you can still preview the output format before you have them set up.

## (Google Colab) two separate requirements

Colab runs on Linux, so it *can* run generation — but two things must be handled, and they're independent:

1. **The repo (code + data).** The notebook imports `scripts/crest_conformers_standalone.py` and locates example data, so the repo must be present. In a separate cell: `!git clone <your-repo-url>` and run this notebook from inside the checkout so `scripts/` and `results/` resolve. *(Repo import alone does **not** make generation work.)*
2. **The `xtb` + `crest` binaries.** These are **not** on Colab by default. The cell below installs them (via `condacolab`, which **restarts the runtime** on first run — after the restart, re-run the cell to finish, then continue). **Without this install, Step 3 cannot generate** — it will print an ACTION REQUIRED message and do nothing.

On a non-Colab machine this cell is a no-op (install `xtb`/`crest` through your own environment — see Requirements).

In [1]:
import sys, shutil, subprocess
IN_COLAB = 'google.colab' in sys.modules

def _sh(cmd):
    print('$', ' '.join(cmd)); subprocess.run(cmd, check=True)

if not IN_COLAB:
    print('Not on Colab — skipping. Install xtb + crest via your own environment (see Requirements above).')
elif shutil.which('xtb') and shutil.which('crest'):
    print('xtb + crest already available:', shutil.which('xtb'), '|', shutil.which('crest'))
else:
    # condacolab bootstraps conda on Colab. The FIRST install() RESTARTS the runtime;
    # after the restart, re-run THIS cell and it will finish installing xtb/crest.
    try:
        import condacolab
        condacolab.check()   # succeeds only once conda is ready (i.e. after the restart)
        _sh(['conda', 'install', '-q', '-y', '-c', 'conda-forge', 'xtb', 'crest', 'rdkit'])
        print('installed →  xtb:', shutil.which('xtb'), '| crest:', shutil.which('crest'))
    except ImportError:
        _sh([sys.executable, '-m', 'pip', 'install', '-q', 'condacolab'])
        import condacolab
        condacolab.install()   # <-- restarts the Colab runtime; then RE-RUN this cell

Not on Colab — skipping. Install xtb + crest via your own environment (see Requirements above).


## Setup

In [2]:
import os, sys, shutil, json
from pathlib import Path

here = Path.cwd()
ROOT = next(p for p in [here, *here.parents] if (p / 'scripts').is_dir() and (p / 'results').is_dir())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'scripts'))

# Standalone generator: SMILES -> CREST -> RMSD-deduplicated diverse unique-conformer set.
import crest_conformers_standalone as cc
print('repo root:', ROOT)
print('default unique conformers per solvent:', cc.DEFAULT_UNIQUE_CONFS)

repo root: C:\Users\Admin\Documents\Hu Lab\Code\Python\Chameleon_Predictor
default unique conformers per solvent: 20


## Step 1 — check the external binaries
The generation needs `xtb` and `crest` on `PATH`. This cell just reports whether they are available on *this* machine.

In [3]:
bins = {b: shutil.which(b) for b in ('xtb', 'crest')}
HAVE_BINARIES = all(bins.values())
for b, p in bins.items():
    print(f'  {b:6}: {p if p else "NOT FOUND"}')
print()
if HAVE_BINARIES:
    print('Ready to generate — paste your SMILES in Step 2 and run Step 3.')
else:
    print('ACTION REQUIRED: xtb and/or crest are not on your PATH.')
    print('Install them, then restart the kernel and re-run this notebook:')
    print('    conda install -c conda-forge xtb crest      # Linux / macOS / WSL')

  xtb   : NOT FOUND
  crest : NOT FOUND

ACTION REQUIRED: xtb and/or crest are not on your PATH.
Install them, then restart the kernel and re-run this notebook:
    conda install -c conda-forge xtb crest      # Linux / macOS / WSL


## Step 2 — enter your molecule
Edit these values. `SMILES` is the only required one; the rest have sensible defaults.

| variable | meaning | default |
|---|---|---|
| `SMILES` | the molecule to sample | *(a DOPC macrocycle example)* |
| `NAME` | label for the run folder / files | `"my_molecule"` |
| `CHARGE` | formal-charge override; `None` = auto from SMILES | `None` |
| `N_THREADS` | CPU cores for xTB/CREST; `None` = all | `None` |
| `MAX_CONFS` | cap on **unique conformers retained per solvent** | `None` (default 20) |
| `OUTDIR` | where the run folder is created | `"results/notebook_runs"` |

In [4]:
# ── paste your SMILES here ─────────────────────────────────────────────
SMILES    = "C#CCCC(=O)N[C@H]1CSCc2ccccc2CSC[C@@H](C(N)=O)NC(=O)[C@H](c2cccs2)NC(=O)[C@H](CO)NC(=O)[C@@H]2CCN2C(=O)[C@H](CO)NC1=O"
NAME      = "my_molecule"
CHARGE    = None          # e.g. -1 for a deprotonated carboxylate; None = auto
N_THREADS = None          # None = use all CPU cores
MAX_CONFS = None          # None = engine default (50)
OUTDIR    = "results/notebook_runs"

# quick sanity check on the SMILES before committing to a long run
from rdkit import Chem
m = Chem.MolFromSmiles(SMILES)
assert m is not None, f'RDKit could not parse SMILES: {SMILES!r}'
print(f'SMILES OK  |  {m.GetNumAtoms()} heavy atoms  |  name = {NAME!r}')

SMILES OK  |  54 heavy atoms  |  name = 'my_molecule'


## Step 3 — generate the ensembles
One call to `generate_ensembles()` runs the full pipeline for both solvents: RDKit ETKDGv3 embedding → xTB pre-optimisation → CREST iMTD-GC sampling → **RMSD deduplication to a diverse unique set (default 20/solvent)** → `ensemble.sdf` (geometries) + `ensemble.json` (geometry set + energy/weight metadata).

This is the compute-heavy step (minutes to hours depending on size and cores). **If `xtb`/`crest` are installed, this cell generates your ensembles.** If they are not yet installed, it tells you exactly what to install and does nothing else — install them (see Step 1), restart the kernel, and re-run.

In [5]:
out = None
if HAVE_BINARIES:
    out = cc.generate_ensembles(
        SMILES, name=NAME, charge=CHARGE, n_threads=N_THREADS,
        max_confs=MAX_CONFS, outdir=OUTDIR,
    )
    print('\nall four ensemble files written:', out['ok'])
    print('run directory:', out['work_dir'])
    for solv in ('water', 'mem'):
        print(f"  {solv}: {out[f'{solv}_n_confs']} unique conformers  "
              f"({Path(out[f'{solv}_sdf']).name}, {Path(out[f'{solv}_json']).name})")
else:
    print('ACTION REQUIRED — xtb/crest not found, so nothing was generated.')
    print('Install xtb + crest (Linux / macOS / WSL), restart the kernel, then re-run')
    print('this cell to generate the water + chloroform ensembles for your SMILES:')
    print('    conda install -c conda-forge xtb crest')

ACTION REQUIRED — xtb/crest not found, so nothing was generated.
Install xtb + crest (Linux / macOS / WSL), restart the kernel, then re-run
this cell to generate the water + chloroform ensembles for your SMILES:
    conda install -c conda-forge xtb crest


## Step 4 — inspect the result
Load the water ensemble that was just produced and check its structure. (If generation was skipped for lack of binaries, we load a pre-computed example so you can still see the exact output format your run will produce.)

In [6]:
import numpy as np

if out and out['ok']:
    water_json = Path(out['water_json'])
    src = 'your run'
else:
    cands = sorted((ROOT / 'results' / 'conformers').rglob('water/ensemble.json'))
    water_json = cands[0]
    src = f'pre-computed example ({water_json.parent.parent.name}); predates dedup'

data = json.load(open(water_json))
confs = data['conformers']
en = np.array([c.get('totalenergy', np.nan) for c in confs], float)
print(f'source            : {src}')
print(f'geometry mode     : {data.get("geometry_mode", "n/a")}')
print(f'unique conformers : {len(confs)}   (fresh runs cap at ~{cc.DEFAULT_UNIQUE_CONFS} by RMSD dedup)')
if np.isfinite(en).any():
    print(f'energy spread     : {(np.nanmax(en) - np.nanmin(en)):.4f} Hartree  (metadata / QC only)')
print(f'per-conformer keys: {list(confs[0].keys())}')
print('\nGeometry is the object of analysis; energies + Boltzmann weights are retained as')
print('metadata/QC only — Notebook 2 summarizes geometry (median/range), not weighted means.')

source            : pre-computed example (6-4-4-13 Xylene Linker); predates dedup
geometry mode     : n/a
unique conformers : 408   (fresh runs cap at ~20 by RMSD dedup)
energy spread     : 0.0095 Hartree  (metadata / QC only)
per-conformer keys: ['totalenergy', 'boltzmannweight', 'psa', 'hbonds', 'relativeenergy', 'conformerweights', 'set', 'degeneracy']

Geometry is the object of analysis; energies + Boltzmann weights are retained as
metadata/QC only — Notebook 2 summarizes geometry (median/range), not weighted means.


## What you get
```
<OUTDIR>/<timestamp>_<NAME>/
├── water/
│   ├── ensemble.sdf     # unique-conformer geometries (implicit water)
│   └── ensemble.json    # same conformers + energy/weight metadata
└── mem/
    ├── ensemble.sdf     # unique-conformer geometries (implicit chloroform)
    └── ensemble.json
```
These feed directly into **Notebook 2** (descriptors) and **Notebook 3** (figures).

**What each file holds.** The **`.sdf`** is the set of RMSD-deduplicated unique conformer geometries — the object of analysis; Notebook 2 recomputes every descriptor (PSA, Rg, IMHB, shape) from them and summarizes their **distribution** (median, IQR, range). The **`.json`** carries the same conformers plus **energy and Boltzmann-weight metadata** (`geometry_mode`, `energy_min/median/max`, per-conformer `totalenergy`/`boltzmannweight`) — retained for QC / an optional sanity check, **not** as the aggregation driver.

> **Note on the raw `psa`/`hbonds` fields** in `ensemble.json`: they use the engine's older v2 convention and are not used downstream — Notebook 2 recomputes the 3D-PSA of record (Ono 2019 / Begnini 2021: N/O + polar H, Ertl sulfur rule).

> **Remaining external dependency:** the only thing preventing notebook-only execution is that `xtb` and `crest` are external binaries (Linux/macOS/WSL). Everything else — SMILES input, validation, generation, output handling — is driven from this notebook.